Test infill gap detection

In [ ]:
from pathlib import Path

# Create a test partial XML file with empty measures
TEST_XML_PATH = "output/test_partial.xml"
EMPTY_MEASURES = [3, 4, 5]  # measures to leave empty (these will be filled)

Path("output").mkdir(exist_ok=True)
create_test_partial_xml(TEST_XML_PATH, empty_measures=EMPTY_MEASURES)
print(f"Created test file: {TEST_XML_PATH}")
print(f"Empty measures: {EMPTY_MEASURES}")

# Verify gaps are detected
score, gaps = load_partial_xml(TEST_XML_PATH)
print(f"Detected gaps: {gaps}")

<hr>

### Generation Params

In [ ]:
INFILL_TEMPERATURE = 0.9
INFILL_TOP_K = 50
INFILL_MAX_NEW = 500

### Test Generation on Dataset Score

In [ ]:
import music21 as m21
from pathlib import Path

# Get a score from the dataset
paths = load_cached_paths()
if not paths:
    print("No dataset found. Run Download Dataset first.")
else:
    # Take the first score
    original_path = paths[0]
    print(f"Using score: {original_path}")
    
    # Parse the score
    original_score = m21.converter.parse(original_path)
    
    # Find how many measures it has
    if original_score.parts:
        measures = list(original_score.parts[0].getElementsByClass(m21.stream.Measure))
        n_measures = len(measures)
        print(f"Original score has {n_measures} measures")
        
        if n_measures >= 6:
            # Create a modified score with measures 3-4 emptied
            modified_score = m21.stream.Score()
            
            for part in original_score.parts:
                new_part = m21.stream.Part()
                for measure in part.getElementsByClass(m21.stream.Measure):
                    m_num = measure.number if measure.number is not None else 0
                    new_measure = m21.stream.Measure(number=m_num)
                    
                    if m_num in [3, 4]:
                        rest = m21.note.Rest(quarterLength=4)
                        rest.style.hideObjectOnPrint = True
                        
                        new_measure.elements = []
                        new_measure.insert(0, rest)
                    else:
                        for el in measure:
                            if isinstance(el, (m21.note.Note, m21.note.Rest, m21.chord.Chord, m21.stream.Voice)):
                                new_measure.insert(el.offset, el)

                    new_part.append(new_measure)
                    
#                     # Only copy content for non-gap measures
#                     if m_num not in [3, 4]:
#                         for el in measure:
#                             if isinstance(el, (m21.note.Note, m21.note.Rest, m21.chord.Chord, m21.stream.Voice)):
#                                 new_measure.insert(el.offset, el)
                    
#                     new_part.append(new_measure)
                modified_score.append(new_part)
            
            # Save the modified score
            modified_path = "output/real_partial.xml"
            modified_score.write("musicxml", modified_path)
            # print(f"Created partial score: {modified_path}")
            # print("Emptied measures: 3, 4")
            
            # Verify gaps detected
            gaps = identify_empty_measures(modified_score)
            score.write("musicxml", "output/what_the_heck.xml")
            print(f"Detected gaps: {gaps}")
            
            # Run infill
            print("\nRunning infill on real score...")
            print("=" * 50)
            result_paths = infill_and_export(
                score=modified_score,
                max_new=INFILL_MAX_NEW,
                temperature=INFILL_TEMPERATURE,
                top_k=INFILL_TOP_K,
                log=print,
                device=DEVICE,
            )
            print("=" * 50)
            
            if result_paths["xml"]:
                print("\n--- Filled Score Preview ---")
                show_text_score(result_paths["xml"], print)
        else:
            print(f"Score too short ({n_measures} measures), need at least 6")